# 4 · Resolved-rate motion control

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rudip1/learn-manipulator-control/blob/main/2_notebooks/exercises/04_resolved_rate_control.ipynb)

Theory: [`1_theory/04_resolved_rate_control.md`](../../1_theory/04_resolved_rate_control.md) · notation:
[`00_notation.md`](../../1_theory/00_notation.md)

In [ ]:
# Install the module when it is not importable yet (Colab, fresh environment).
import importlib.util
if importlib.util.find_spec("manipulator_control") is None:
    %pip install -q git+https://github.com/Rudip1/learn-manipulator-control

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import manipulator_control as mc
from manipulator_control import plotting

plt.rcParams.update({"figure.dpi": 80, "axes.grid": True})
np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(4)

## Recap

- Task error $\tilde\sigma = \sigma_d - \sigma$ (eq. 4.2); control law $\dot q = J^\#(\dot\sigma_d + K\tilde\sigma)$
  (eq. 4.3) with $J^\# \in \{J^T, J^\dagger, J^\dagger_\lambda\}$.
- Pseudoinverse (eq. 4.4): minimum-norm joint rates; with full row rank $\dot{\tilde\sigma} = -K\tilde\sigma$ exactly,
  so the error decays as $e^{-Kt}$ along a straight line (eq. 4.5).
- Transpose: gradient descent on $\tfrac12\|\tilde\sigma\|^2$, never increases the error, rate $k s_i^2$ per singular
  direction (eq. 4.7).
- Without feedforward a moving target is tracked with lag $K^{-1}\dot\sigma_d$ (eq. 4.8).
- In discrete time the error factor is $1 - k\Delta t$: stable only for $0 < k\Delta t < 2$ (eq. 4.9).
- Orientation error $\log(R_dR^T)$ pairs with $J_O$ and also decays as $e^{-kt}$ (eq. 4.10).

## Three inverses on a planar two-link arm

The arm starts at $q = (0.2, 0.5)$ and must bring its tip to $(0, 1)$; $k = 1$, $\Delta t = 1/60$ s. The whole loop
runs in C++ (`simulate_resolved_rate`); Python only plots.

In [ ]:
arm = mc.robots.planar([0.75, 0.5])
q0, goal = np.array([0.2, 0.5]), mc.translation([0.0, 1.0, 0.0])
methods = {"transpose": mc.InverseMethod.Transpose, "pseudoinverse": mc.InverseMethod.Pseudoinverse,
           "DLS": mc.InverseMethod.DampedLeastSquares}
runs = {}
for name, method in methods.items():
    cfg = mc.ResolvedRateConfig(method=method, space=mc.TaskSpace.PlanarPosition, gain=1.0, damping=0.1,
                                dt=1 / 60, steps=600)
    runs[name] = mc.simulate_resolved_rate(arm, q0, goal, cfg)

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
for k, (name, traj) in enumerate(runs.items()):
    if name == "pseudoinverse":
        for j in range(0, 601, 60):
            plotting.draw_planar_arm(ax[0], arm.frames(traj.q[j]), color=f"C{k}", alpha=0.15, linewidth=1.5)
    ax[0].plot(traj.ee_position[:, 0], traj.ee_position[:, 1], color=f"C{k}", label=name)
    ax[1].semilogy(traj.t, traj.error_norm, color=f"C{k}", label=name)
    ax[2].plot(traj.t, np.linalg.norm(traj.qdot, axis=1), color=f"C{k}", label=name)
ax[0].plot(0, 1, "kx", ms=10); plotting.planar_axes(ax[0], limit=1.4, title="paths (pseudoinverse snapshots)")
ax[1].semilogy(traj.t, runs["pseudoinverse"].error_norm[0] * np.exp(-traj.t), "k:", label=r"$e^{-kt}$")
ax[1].set_xlabel("t [s]"); ax[1].set_title("task error"); ax[1].legend()
ax[2].set_xlabel("t [s]"); ax[2].set_title(r"$\|\dot q\|$ [rad/s]"); ax[2].legend()
plt.tight_layout(); plt.show()

## Full pose of a six-joint arm

For the PUMA 560 the task is the whole pose: position error plus the axis–angle error $\log(R_dR^T)$. Both decay
at the same exponential rate.

In [ ]:
puma = mc.robots.puma560()
q_start = np.array([0.1, 0.6, -0.4, 0.3, 0.8, 0.2])
T_goal = puma.end_effector(np.array([0.5, 0.3, -0.1, -0.2, 1.1, -0.4]))
cfg = mc.ResolvedRateConfig(space=mc.TaskSpace.Pose, gain=2.0, dt=0.01, steps=400)
traj = mc.simulate_resolved_rate(puma, q_start, T_goal, cfg)
plt.figure(figsize=(5.5, 3.5))
plt.semilogy(traj.t, np.linalg.norm(traj.error[:, :3], axis=1), label="position error [m]")
plt.semilogy(traj.t, np.linalg.norm(traj.error[:, 3:], axis=1), label="orientation error [rad]")
plt.semilogy(traj.t, np.linalg.norm(traj.error[0, 3:]) * np.exp(-2 * traj.t), "k:", label=r"$e^{-2t}$")
plt.xlabel("t [s]"); plt.legend(); plt.title("PUMA 560, pose control, $k = 2$"); plt.show()

## Tracking a circle

The planar arm follows a circle of radius $r = 0.3$ m at $\omega = 0.5$ rad/s, with and without the feedforward
term. Without it the error obeys $\dot{\tilde\sigma} = \dot\sigma_d - k\tilde\sigma$. For a constant reference velocity
that settles at $\dot\sigma_d/k$ (eq. 4.8); here $\dot\sigma_d$ rotates at $\omega$, and the same equation, solved for
a rotating input, gives a lag of $r\omega/\sqrt{k^2 + \omega^2}$ — slightly less than $r\omega/k$.

In [ ]:
dt, k = 0.01, 2.0
t = np.arange(0, 15 + dt / 2, dt)
centre, radius, w = np.array([0.6, 0.4]), 0.3, 0.5
pos = centre + radius * np.column_stack([np.cos(w * t), np.sin(w * t)])
vel = radius * w * np.column_stack([-np.sin(w * t), np.cos(w * t)])
q_c = np.array([-0.3, 1.6])
cfg = mc.ResolvedRateConfig(gain=k, dt=dt)
with_ff = mc.simulate_position_tracking(arm, q_c, pos, vel, cfg)
without_ff = mc.simulate_position_tracking(arm, q_c, pos, np.zeros_like(vel), cfg)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(pos[:, 0], pos[:, 1], "k--", label="reference")
ax[0].plot(with_ff.ee_position[:, 0], with_ff.ee_position[:, 1], label="with feedforward")
ax[0].plot(without_ff.ee_position[:, 0], without_ff.ee_position[:, 1], label="without")
plotting.draw_planar_arm(ax[0], arm.frames(q_c), color="0.6")
plotting.planar_axes(ax[0]); ax[0].set_xlim(-0.2, 1.2); ax[0].set_ylim(-0.3, 1.0); ax[0].legend(fontsize=8)
ax[1].plot(t, with_ff.error_norm, label="with feedforward")
ax[1].plot(t, without_ff.error_norm, label="without")
ax[1].axhline(radius * w / np.hypot(k, w), color="k", linestyle=":", label=r"$r\omega/\sqrt{k^2+\omega^2}$")
ax[1].set_xlabel("t [s]"); ax[1].set_ylabel("tracking error [m]"); ax[1].legend()
plt.tight_layout(); plt.show()

## ✏️ Exercises

### ✏️ 1. A pseudoinverse from the SVD

Implement eq. (4.4): invert the singular values above `tol * s_max`, zero the rest. It must also work for
rank-deficient matrices.

In [ ]:
def my_pinv(J, tol=1e-10):
    # ✏️ your code here
    raise NotImplementedError("exercise")

In [ ]:
A = rng.normal(size=(3, 5))
B = rng.normal(size=(4, 6)); B[3] = B[0] + B[1]
for M in (A, A.T, B):
    assert np.allclose(my_pinv(M), mc.pseudoinverse(M))
print("✓ my_pinv")

### ✏️ 2. Write the control loop yourself

Implement resolved-rate control of the planar arm with the pseudoinverse, using only `arm.end_effector`,
`mc.geometric_jacobian` and `my_pinv`. Return the joint trajectory as an array of shape `(steps + 1, 2)`.

In [ ]:
def my_resolved_rate(arm, q0, p_goal, gain, dt, steps):
    # ✏️ your code here
    raise NotImplementedError("exercise")

In [ ]:
Q = my_resolved_rate(arm, q0, np.array([0.0, 1.0]), gain=1.0, dt=1 / 60, steps=600)
assert np.allclose(Q, runs["pseudoinverse"].q, atol=1e-9)
print("✓ my_resolved_rate reproduces the C++ trajectory")

### ✏️ 3. Predict the tracking lag

The reference moves on a straight line at constant velocity $v = (0.04, -0.03)$ m/s, the gain is $k = 2.5$ and
there is no feedforward. Predict the steady-state error vector, then check it against the simulation.

In [ ]:
v, k3 = np.array([0.04, -0.03]), 2.5
# ✏️ your code here
raise NotImplementedError("exercise")

In [ ]:
t3 = np.arange(0, 6 + 0.005, 0.005)
p_start = arm.end_effector(np.array([0.3, 1.2]))[:2, 3]
ref = p_start + t3[:, None] * v
tr = mc.simulate_position_tracking(arm, np.array([0.3, 1.2]), ref, np.zeros_like(ref),
                                   mc.ResolvedRateConfig(gain=k3, dt=0.005))
assert np.allclose(tr.error[-1], e_ss_predicted, atol=1e-4)
print("✓ steady-state error", tr.error[-1])
print(f"circle: measured lag {without_ff.error_norm[-1]:.4f} m, predicted {radius * w / np.hypot(k, w):.4f} m")

### ✏️ 4. The transpose method's gain limit

Near a configuration $q$, the transpose method multiplies the error component along $u_i$ by $1 - k\Delta t\,s_i^2$
per step (section 4.6). Compute the largest gain that keeps the iteration stable at $q = (0.2, 0.5)$ with
$\Delta t = 1/60$ s.

In [ ]:
dt4 = 1 / 60
# ✏️ your code here
raise NotImplementedError("exercise")

In [ ]:
p_here = arm.end_effector(q0)[:2, 3]
tiny_goal = mc.translation([p_here[0] + 1e-4, p_here[1] + 1e-4, 0])
def grows(gain):
    tr = mc.simulate_resolved_rate(arm, q0, tiny_goal, mc.ResolvedRateConfig(
        method=mc.InverseMethod.Transpose, space=mc.TaskSpace.PlanarPosition, gain=gain, dt=dt4, steps=300))
    return tr.error_norm[-1] > tr.error_norm[0]
assert not grows(0.95 * k_max) and grows(1.05 * k_max)
print(f"✓ k_max = {k_max:.1f}")

## 🔨 Break it

### 🔨 1. Gain too high for the control period

With the pseudoinverse the per-step factor is $1 - k\Delta t$ (eq. 4.9). Watch the error for several values of
$k\Delta t$: monotone, one-step, oscillating, divergent.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
for kdt in [0.3, 1.0, 1.6, 1.95, 2.05]:
    cfg = mc.ResolvedRateConfig(space=mc.TaskSpace.PlanarPosition, gain=kdt * 60, dt=1 / 60, steps=40)
    tr = mc.simulate_resolved_rate(arm, q0, tiny_goal, cfg)
    ax.semilogy(np.arange(41), tr.error_norm, "o-", ms=3, label=f"$k\\Delta t$ = {kdt}")
ax.set_xlabel("step"); ax.set_ylabel("error [m]"); ax.legend(fontsize=8); ax.set_title("discrete-time gain limit")
plt.show()

### 🔨 2. The wrong sign

Define the error as $\sigma - \sigma_d$ and the controller pushes away from the target, exponentially.

In [ ]:
q = q0.copy(); err = []
for _ in range(120):
    e_wrong = arm.end_effector(q)[:2, 3] - np.array([0.0, 1.0])  # sign flipped
    err.append(np.linalg.norm(e_wrong))
    q = q + (1 / 60) * mc.pseudoinverse(mc.geometric_jacobian(arm, q)[:2]) @ e_wrong
plt.figure(figsize=(5, 3)); plt.plot(np.arange(120) / 60, err); plt.xlabel("t [s]"); plt.ylabel("distance to goal [m]")
plt.title("error with the wrong sign"); plt.show()

### 🔨 3. A target out of reach

Ask for $(1.5, 0)$, beyond the reach $l_1 + l_2 = 1.25$ m. The arm stretches out towards the boundary, where it
becomes singular: the pseudoinverse commands ever larger joint rates for an error it cannot remove, while DLS stays
calm. Chapter 5 explains why.

In [ ]:
far = mc.translation([1.5, 0.0, 0.0])
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for name in ["pseudoinverse", "DLS"]:
    cfg = mc.ResolvedRateConfig(method=methods[name], space=mc.TaskSpace.PlanarPosition, gain=1.0, damping=0.1,
                                dt=1 / 60, steps=600)
    tr = mc.simulate_resolved_rate(arm, np.array([0.3, 0.6]), far, cfg)
    ax[0].plot(tr.t, tr.error_norm, label=name)
    ax[1].semilogy(tr.t[:-1], np.abs(tr.qdot[:-1]).max(axis=1), label=name)
ax[0].set_title("distance to the target [m]"); ax[0].set_xlabel("t [s]"); ax[0].legend()
ax[1].set_title(r"max $|\dot q_i|$ [rad/s]"); ax[1].set_xlabel("t [s]"); ax[1].legend()
plt.tight_layout(); plt.show()

## What to remember

- Resolved-rate control: $\dot q = J^\#(\dot\sigma_d + K\tilde\sigma)$, recomputed every control period.
- The pseudoinverse makes the task error decay as $e^{-Kt}$ along a straight line; the transpose is cheaper and
  never increases the error, but converges slowly along weak directions.
- Use feedforward when the target moves; otherwise expect a lag of $K^{-1}\dot\sigma_d$.
- What limits the gain is $k\Delta t < 2$ (in practice far less), not $k$ alone.
- For orientation use $\log(R_dR^T)$ with the angular-velocity rows of the geometric Jacobian.